# CNN-GRU Electricity Price Forecasting
### Sequential Cascade: Generation → Load → Price (Day-Ahead & Real-Time)

**Four-stage forecasting pipeline:**

| Stage | Target | Model | Inputs |
|-------|--------|-------|--------|
| 1a | `gen_solar` | Day-Ahead (24 hr) | Base historical features |
| 1b | `gen_wind` | Day-Ahead (24 hr) | Base + gen_solar predictions |
| 2 | `total load actual` | Day-Ahead (24 hr) | Base + gen_solar + gen_wind predictions |
| 3 | `price day ahead` | Day-Ahead (24 hr) | Base + gen_solar + gen_wind + load predictions |
| 4 | `price actual` | Real-Time (1 hr) | Stream 1: Base + cascade predictions (encoder) |
| | | | Stream 2: TSO published forecasts + day-ahead price (dense head) |

**Stream 2 rationale (Option B):** The real-time model receives both upstream cascade
predictions (via Stream 1 / encoder) AND published TSO forecasts (via Stream 2 / dense head).
These are complementary signals — TSO forecasts use meteorological models and grid data
the cascade model does not see. The model learns to weight both sources during training.

**Training discipline:**
- Cascade predictions appended to X using the *trained model's outputs*, not actuals,
  so the price model trains on the same quality of upstream forecast it sees at evaluation time.
- Val set used for hyperparameter tuning only.
- Test set evaluated once after hyperparameters are locked.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import os
import copy

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

## 2. Load & Inspect Data

In [ ]:
energy_weather = pd.read_csv('../../data/processed/energy_weather_merged.csv')
energy_weather.info()
energy_weather.head()

## 3. Column Definitions

In [ ]:
# ── Pipeline targets (in forecast order) ─────────────────────────────────────
GENERATION_TARGETS = ['gen_solar', 'gen_wind']
LOAD_TARGETS       = ['total load actual']
PRICE_DA_TARGET    = 'price day ahead'
PRICE_RT_TARGET    = 'price actual'
ALL_TARGETS        = GENERATION_TARGETS + LOAD_TARGETS + [PRICE_DA_TARGET, PRICE_RT_TARGET]

# ── Cascade order for stages 1a, 1b, 2 ───────────────────────────────────────
# These targets are trained first and their predictions appended to X
# before the price models are trained.
CASCADE_TARGETS = GENERATION_TARGETS + LOAD_TARGETS   # ['gen_solar', 'gen_wind', 'total load actual']

# ── Published TSO / ENTSO-E forecast columns (Stream 2, real-time model) ─────
# These represent what is genuinely available before real-time delivery:
#   - Day-ahead generation and load forecasts published by TSOs
#   - Settled day-ahead price (always known before real-time settlement)
# Option B: both TSO forecasts AND upstream cascade predictions are used.
#   TSO forecasts → Stream 2 (dense head)
#   Cascade predictions → Stream 1 (encoder, via appended X columns)
REALTIME_FORECAST_COLS = [
    'forecast solar day ahead',
    'forecast wind onshore day ahead',
    'total load forecast',
    'price day ahead',
]

IDENTIFIER_COLS = ['time', 'date', 'city_name', 'hour']

print("Pipeline targets (in order):")
for i, t in enumerate(ALL_TARGETS, 1):
    tag = '(cascade → appended to X)' if t in CASCADE_TARGETS else '(terminal — price model)'
    print(f"  {i}. {t:<25} {tag}")
print(f"\nReal-time Stream 2 features: {REALTIME_FORECAST_COLS}")

## 4. Feature Engineering

In [ ]:
energy_weather['time'] = pd.to_datetime(energy_weather['time'], utc=True)
energy_weather['date'] = pd.to_datetime(energy_weather['date'])
energy_weather = energy_weather.sort_values('time').reset_index(drop=True)

# ── Cyclical time features ────────────────────────────────────────────────────
energy_weather['hour_sin']    = np.sin(2 * np.pi * energy_weather['hour'] / 24)
energy_weather['hour_cos']    = np.cos(2 * np.pi * energy_weather['hour'] / 24)

energy_weather['day_of_week'] = energy_weather['date'].dt.dayofweek
energy_weather['day_sin']     = np.sin(2 * np.pi * energy_weather['day_of_week'] / 7)
energy_weather['day_cos']     = np.cos(2 * np.pi * energy_weather['day_of_week'] / 7)

energy_weather['month']       = energy_weather['date'].dt.month
energy_weather['month_sin']   = np.sin(2 * np.pi * energy_weather['month'] / 12)
energy_weather['month_cos']   = np.cos(2 * np.pi * energy_weather['month'] / 12)

energy_weather['is_weekend']  = (energy_weather['day_of_week'] >= 5).astype(int)
energy_weather['year']        = energy_weather['date'].dt.year

# ── Lag features for all targets ─────────────────────────────────────────────
# Short lags (1-3 hr): explicit recent-change signals
# Day lags (24, 48 hr): same-hour yesterday / two-days-ago patterns
LAG_HOURS = [1, 2, 3, 24, 48]

for target in ALL_TARGETS:
    col = target.replace(' ', '_')
    for lag in LAG_HOURS:
        energy_weather[f'{col}_lag_{lag}'] = energy_weather[target].shift(lag)

energy_weather = energy_weather.dropna().reset_index(drop=True)
print(f"Shape after feature engineering: {energy_weather.shape}")

## 5. Feature Column Lists

`BASE_HIST_FEATURE_COLS` defines the encoder input for Stage 1 models.
Each subsequent stage automatically widens this set by appending upstream
predictions via `append_cascade_to_fold()` — no manual changes needed here.

In [ ]:
SIN_COS_COLS = [
    'hour_sin', 'hour_cos',
    'day_sin',  'day_cos',
    'month_sin','month_cos',
    'is_weekend',
]

LAG_COLS = [
    f"{t.replace(' ', '_')}_lag_{lag}"
    for t in ALL_TARGETS
    for lag in LAG_HOURS
]

# ── Base historical features (Stage 1 encoder input) ─────────────────────────
BASE_HIST_FEATURE_COLS = [
    'gen_hydro', 'gen_pumped_hydro', 'gen_fossil', 'gen_nuclear', 'gen_other',
    'temp_Barcelona',      'temp_Bilbao',      'temp_Madrid',      'temp_Seville',      'temp_Valencia',
    'pressure_Barcelona',  'pressure_Bilbao',  'pressure_Madrid',  'pressure_Seville',  'pressure_Valencia',
    'humidity_Barcelona',  'humidity_Bilbao',  'humidity_Madrid',  'humidity_Seville',  'humidity_Valencia',
    'wind_speed_Barcelona','wind_speed_Bilbao','wind_speed_Madrid','wind_speed_Seville','wind_speed_Valencia',
    'wind_deg_Barcelona',  'wind_deg_Bilbao',  'wind_deg_Madrid',  'wind_deg_Seville',  'wind_deg_Valencia',
    'rain_1h_Barcelona',   'rain_1h_Bilbao',   'rain_1h_Madrid',   'rain_1h_Seville',   'rain_1h_Valencia',
    'rain_3h_Barcelona',   'rain_3h_Bilbao',   'rain_3h_Madrid',   'rain_3h_Seville',   'rain_3h_Valencia',
    'snow_3h_Barcelona',   'snow_3h_Bilbao',   'snow_3h_Madrid',   'snow_3h_Seville',   'snow_3h_Valencia',
    'clouds_all_Barcelona','clouds_all_Bilbao','clouds_all_Madrid','clouds_all_Seville','clouds_all_Valencia',
    'temp_avg',
] + SIN_COS_COLS + LAG_COLS

# Columns that need MinMaxScaler (sin/cos already bounded [-1,1])
BASE_SCALE_COLS   = [c for c in BASE_HIST_FEATURE_COLS if c not in SIN_COS_COLS]
BASE_NOSCALE_COLS = SIN_COS_COLS

print(f"Base historical features : {len(BASE_HIST_FEATURE_COLS)}")
print(f"  → scaled               : {len(BASE_SCALE_COLS)}")
print(f"  → not scaled (sin/cos) : {len(BASE_NOSCALE_COLS)}")
print(f"Real-time Stream 2 cols  : {len(REALTIME_FORECAST_COLS)} → {REALTIME_FORECAST_COLS}")

## 6. Year-Aligned Walk-Forward Cross-Validation

| Split | Length | Purpose |
|-------|--------|---------|
| Train | 2 years | Model fitting |
| Val   | 1 year  | Hyperparameter tuning |
| Test  | 1 year  | Final evaluation (touch once only) |

Each fold stores:
- `X_train/val/test_base` — immutable base arrays; reset before each HP config
- `X_train/val/test` — working copies extended with cascade predictions
- `Xfc_train/val/test` — Stream 2 (TSO forecasts + day-ahead price) for real-time model
- `cascade_preds` — dict of raw predictions per target, populated as pipeline runs

In [ ]:
all_years = sorted(energy_weather['year'].unique())
print(f"Available years: {all_years}")

TRAIN_YEARS = 2
VAL_YEARS   = 1
TEST_YEARS  = 1
FOLD_STEP   = 1

folds = []

for fold_num, start in enumerate(
    range(0, len(all_years) - TRAIN_YEARS - VAL_YEARS - TEST_YEARS + 1, FOLD_STEP),
    start=1
):
    train_yrs = all_years[start                              : start + TRAIN_YEARS]
    val_yrs   = all_years[start + TRAIN_YEARS                : start + TRAIN_YEARS + VAL_YEARS]
    test_yrs  = all_years[start + TRAIN_YEARS + VAL_YEARS    : start + TRAIN_YEARS + VAL_YEARS + TEST_YEARS]

    df_train = energy_weather[energy_weather['year'].isin(train_yrs)].copy()
    df_val   = energy_weather[energy_weather['year'].isin(val_yrs)].copy()
    df_test  = energy_weather[energy_weather['year'].isin(test_yrs)].copy()

    # ── Scale base historical features (fit on train only) ────────────────────
    scaler_X = MinMaxScaler()
    scaler_X.fit(df_train[BASE_SCALE_COLS])

    def _build_X(df):
        scaled    = scaler_X.transform(df[BASE_SCALE_COLS])
        noscale   = df[BASE_NOSCALE_COLS].values
        return np.hstack([scaled, noscale]).astype(np.float32)

    X_train_base = _build_X(df_train)
    X_val_base   = _build_X(df_val)
    X_test_base  = _build_X(df_test)

    # ── Scale Stream 2 (TSO forecasts + day-ahead price) ─────────────────────
    # Fit on train only; apply to all splits.
    # These go into the real-time model's dense head, not the encoder.
    scaler_fc = MinMaxScaler()
    scaler_fc.fit(df_train[REALTIME_FORECAST_COLS])

    Xfc_train = scaler_fc.transform(df_train[REALTIME_FORECAST_COLS]).astype(np.float32)
    Xfc_val   = scaler_fc.transform(df_val[REALTIME_FORECAST_COLS]).astype(np.float32)
    Xfc_test  = scaler_fc.transform(df_test[REALTIME_FORECAST_COLS]).astype(np.float32)

    # ── Scale each target individually (fit on train only) ────────────────────
    target_scalers, y_train_d, y_val_d, y_test_d = {}, {}, {}, {}
    for target in ALL_TARGETS:
        sc = MinMaxScaler()
        sc.fit(df_train[[target]])
        y_train_d[target]   = sc.transform(df_train[[target]]).astype(np.float32)
        y_val_d[target]     = sc.transform(df_val[[target]]).astype(np.float32)
        y_test_d[target]    = sc.transform(df_test[[target]]).astype(np.float32)
        target_scalers[target] = sc

    folds.append({
        'fold'            : fold_num,
        'train_years'     : train_yrs,
        'val_years'       : val_yrs,
        'test_years'      : test_yrs,
        # Immutable base arrays — reset before each HP config
        'X_train_base'    : X_train_base,
        'X_val_base'      : X_val_base,
        'X_test_base'     : X_test_base,
        # Working arrays extended by append_cascade_to_fold()
        'X_train'         : X_train_base.copy(),
        'X_val'           : X_val_base.copy(),
        'X_test'          : X_test_base.copy(),
        # Stream 2 — fixed, never modified
        'Xfc_train'       : Xfc_train,
        'Xfc_val'         : Xfc_val,
        'Xfc_test'        : Xfc_test,
        # Targets
        'y_train'         : y_train_d,
        'y_val'           : y_val_d,
        'y_test'          : y_test_d,
        # Scalers
        'scaler_X'        : scaler_X,
        'scaler_forecast' : scaler_fc,
        'target_scalers'  : target_scalers,
        # Timestamps
        'train_dates'     : df_train['time'].values,
        'val_dates'       : df_val['time'].values,
        'test_dates'      : df_test['time'].values,
        # Populated as cascade runs; reset with X arrays each HP config
        'cascade_preds'   : {},
    })

    print(f"Fold {fold_num}:")
    print(f"  Train : {train_yrs}  ({len(df_train):,} rows)")
    print(f"  Val   : {val_yrs}   ({len(df_val):,} rows)")
    print(f"  Test  : {test_yrs}   ({len(df_test):,} rows)")

print(f"\n{len(folds)} fold(s) created")
print(f"Base X width : {folds[0]['X_train_base'].shape[1]} features")
print(f"Stream 2 width: {folds[0]['Xfc_train'].shape[1]} features")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 2 + len(folds) * 1.2))
colors = {'Train': 'steelblue', 'Val': 'orange', 'Test': 'seagreen'}

for fd in folds:
    f = fd['fold']
    segments = [
        (fd['train_dates'].min(), fd['train_dates'].max(), 'steelblue', 'Train'),
        (fd['val_dates'].min(),   fd['val_dates'].max(),   'orange',    'Val'),
        (fd['test_dates'].min(),  fd['test_dates'].max(),  'seagreen',  'Test'),
    ]
    for left, right, color, label in segments:
        left, right = pd.Timestamp(left), pd.Timestamp(right)
        ax.barh(f, (right - left).days, left=left, color=color, alpha=0.75, height=0.5)
        ax.text(left, f, f'  {label}', va='center', fontsize=8,
                color='white', fontweight='bold')

ax.xaxis_date()
fig.autofmt_xdate()
ax.set_xlabel('Date'); ax.set_ylabel('Fold')
ax.set_title('Year-Aligned Walk-Forward CV — 2yr Train / 1yr Val / 1yr Test')
ax.legend(handles=[mpatches.Patch(color=c, label=l) for l, c in colors.items()],
          loc='lower right')
plt.tight_layout(); plt.show()

## 7. Global Settings & Device

In [ ]:
device = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f"Device: {device}")

WINDOW_SIZE = 168   # 1 week of hourly history (encoder input)
HORIZON_DA  = 24    # generation, load, day-ahead price → predict 24 hrs
HORIZON_RT  = 1     # real-time price → predict next 1 hr

BATCH_SIZE  = 256 if (torch.cuda.is_available() or torch.backends.mps.is_available()) else 64
NUM_WORKERS = 4   if  torch.cuda.is_available() else 0
PIN_MEMORY  = torch.cuda.is_available()

# Derived sizes — read dynamically per stage since X grows
INPUT_SIZE_BASE = folds[0]['X_train_base'].shape[1]
INPUT_SIZE_FC   = folds[0]['Xfc_train'].shape[1]     # Stream 2 width (fixed)
NUM_LAYERS      = 2
DROPOUT         = 0.3

print(f"Window size       : {WINDOW_SIZE} hrs")
print(f"Day-ahead horizon : {HORIZON_DA} hrs")
print(f"Real-time horizon : {HORIZON_RT} hr")
print(f"Base input size   : {INPUT_SIZE_BASE}")
print(f"Stream 2 size     : {INPUT_SIZE_FC}  (TSO forecasts + day-ahead price)")
print(f"Batch size        : {BATCH_SIZE}")

## 8. Dataset Classes

Both classes accept whatever width `X_hist` array they receive, so they work
identically at every cascade stage — no modifications needed as X grows.

**`DayAheadDataset`** — Stages 1a, 1b, 2, 3  
Slides a 168-hour window over X and pairs it with the next 24-hour target block.

**`RealTimeDataset`** — Stage 4 only  
Same 168-hour window from X (Stream 1, encoder), plus a single-row vector of
TSO forecast features for the target hour (Stream 2, dense head).
The cascade predictions from Stages 1–2 arrive via Stream 1 because they were
appended to X by `append_cascade_to_fold()` before this dataset is built.
The TSO forecasts arrive via Stream 2 from `Xfc_train/val/test`.
Both streams are complementary: TSO forecasts carry meteorological / grid signal
the cascade model does not see; the model learns to weight both during training.

In [ ]:
class DayAheadDataset(Dataset):
    """
    Used for: gen_solar (Stage 1a), gen_wind (Stage 1b),
              total load actual (Stage 2), price day ahead (Stage 3).

    Each sample:
      x_hist     : (168, n_features)  — sliding window of historical features.
                   n_features grows at each stage as cascade columns are appended.
      y_next_day : (24,)              — next 24 hourly values of the target.
    """
    def __init__(self, X_hist, y, window_size=168, horizon=24):
        self.X   = torch.FloatTensor(X_hist)
        self.y   = torch.FloatTensor(y).squeeze(-1)
        self.win = window_size
        self.hor = horizon

    def __len__(self):
        # Every valid starting position that has a full window + horizon ahead
        return len(self.X) - self.win - self.hor + 1

    def __getitem__(self, idx):
        x_hist     = self.X[idx : idx + self.win]                       # (168, n_feat)
        y_next_day = self.y[idx + self.win : idx + self.win + self.hor] # (24,)
        return x_hist, y_next_day


class RealTimeDataset(Dataset):
    """
    Used for: price actual (Stage 4).

    Stream 1 — X_hist (encoder):
        Last 168 hours of the extended feature matrix.
        By Stage 4, this already contains appended cascade predictions
        for gen_solar, gen_wind, and total load actual from Stages 1-2.
        These are the model's own best estimates — not TSO actuals —
        so training matches inference conditions exactly.

    Stream 2 — X_forecast (dense head):
        Published TSO forecast features for the single target hour t:
          [forecast solar day ahead,
           forecast wind onshore day ahead,
           total load forecast,
           price day ahead]
        Comes from Xfc_train/val/test (fixed, never modified by cascade).
        Complements Stream 1: TSO forecasts carry meteorological and grid
        information the upstream models do not have access to.

    Target: price actual at the next single hour t.
    """
    def __init__(self, X_hist, X_forecast, y, window_size=168):
        self.X_hist     = torch.FloatTensor(X_hist)
        self.X_forecast = torch.FloatTensor(X_forecast)
        self.y          = torch.FloatTensor(y).squeeze(-1)
        self.win        = window_size

    def __len__(self):
        return len(self.X_hist) - self.win

    def __getitem__(self, idx):
        x_hist       = self.X_hist[idx : idx + self.win]  # (168, n_feat — extended)
        x_forecast_t = self.X_forecast[idx + self.win]    # (4,) — TSO forecasts
        y_t          = self.y[idx + self.win]              # scalar
        return x_hist, x_forecast_t, y_t

In [ ]:
def _loader_kwargs(batch_size, num_workers, pin_memory):
    return dict(batch_size=batch_size, shuffle=False,
                num_workers=num_workers, pin_memory=pin_memory,
                persistent_workers=(num_workers > 0))


def make_loaders_dayahead(fold_data, target,
                           window_size=WINDOW_SIZE, horizon=HORIZON_DA,
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                           pin_memory=PIN_MEMORY):
    """
    Build train/val/test loaders for a day-ahead target.

    Uses fold_data['X_train/val/test'] at call time, so calling this
    AFTER append_cascade_to_fold() automatically gives the next stage
    access to the wider (cascade-extended) feature matrix.
    """
    kw = _loader_kwargs(batch_size, num_workers, pin_memory)
    tr = DataLoader(
        DayAheadDataset(fold_data['X_train'], fold_data['y_train'][target],
                        window_size, horizon),
        drop_last=True, **kw)
    va = DataLoader(
        DayAheadDataset(fold_data['X_val'], fold_data['y_val'][target],
                        window_size, horizon),
        drop_last=False, **kw)
    te = DataLoader(
        DayAheadDataset(fold_data['X_test'], fold_data['y_test'][target],
                        window_size, horizon),
        drop_last=False, **kw)
    return tr, va, te


def make_loaders_realtime(fold_data,
                           window_size=WINDOW_SIZE,
                           batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                           pin_memory=PIN_MEMORY):
    """
    Build train/val/test loaders for the real-time price model.

    Stream 1 (X_hist)    : fold_data['X_train/val/test'] — extended by cascade
    Stream 2 (X_forecast): fold_data['Xfc_train/val/test'] — fixed TSO forecasts

    By the time this is called (after Stages 1-2 have run and appended their
    predictions), X_hist already contains cascade columns for gen_solar,
    gen_wind, and total load actual alongside the base features.
    Xfc is fixed and unmodified throughout the pipeline.
    """
    kw = _loader_kwargs(batch_size, num_workers, pin_memory)
    tr = DataLoader(
        RealTimeDataset(fold_data['X_train'], fold_data['Xfc_train'],
                        fold_data['y_train'][PRICE_RT_TARGET], window_size),
        drop_last=True, **kw)
    va = DataLoader(
        RealTimeDataset(fold_data['X_val'], fold_data['Xfc_val'],
                        fold_data['y_val'][PRICE_RT_TARGET], window_size),
        drop_last=False, **kw)
    te = DataLoader(
        RealTimeDataset(fold_data['X_test'], fold_data['Xfc_test'],
                        fold_data['y_test'][PRICE_RT_TARGET], window_size),
        drop_last=False, **kw)
    return tr, va, te


# ── Quick shape check (on base arrays before any cascade) ─────────────────────
fd = folds[0]
tr, va, te = make_loaders_dayahead(fd, 'gen_solar')
x, y = next(iter(tr))
print(f"DayAheadDataset  x: {tuple(x.shape)}  y: {tuple(y.shape)}")
print(f"  → x expected   : (batch, 168, {INPUT_SIZE_BASE})")
print(f"  → y expected   : (batch, 24)")

tr_rt, va_rt, te_rt = make_loaders_realtime(fd)
xh, xfc, yt = next(iter(tr_rt))
print(f"\nRealTimeDataset  x_hist: {tuple(xh.shape)}  x_fc: {tuple(xfc.shape)}  y: {tuple(yt.shape)}")
print(f"  → x_hist expected     : (batch, 168, {INPUT_SIZE_BASE})")
print(f"  → x_fc expected       : (batch, {INPUT_SIZE_FC})  [TSO forecasts + DA price]")
print(f"  → y expected          : (batch,)")

## 9. Model Architecture

All four stages share `SharedEncoder` (CNN → Bidirectional GRU → Attention).
The encoder `input_size` is read fresh from `fold_data['X_train'].shape[1]`
before each model is instantiated, so it automatically reflects the growing
feature width as cascade columns are appended.

`AttentionLayer` receives `hidden_size * 2` because the GRU is bidirectional.

In [ ]:
class CNNEncoder(nn.Module):
    """
    Three-layer 1-D CNN.  BatchNorm replaces MaxPool so all 168
    timesteps are preserved for the GRU.
    Accepts any input_size — adapts automatically as X widens.
    """
    def __init__(self, input_size, cnn_channels=(64, 128, 128)):
        super().__init__()
        self.out_channels = cnn_channels[-1]
        self.net = nn.Sequential(
            nn.Conv1d(input_size,       cnn_channels[0], kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels[0]), nn.ReLU(),
            nn.Conv1d(cnn_channels[0],  cnn_channels[1], kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_channels[1]), nn.ReLU(),
            nn.Conv1d(cnn_channels[1],  cnn_channels[2], kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_channels[2]), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x.permute(0, 2, 1)).permute(0, 2, 1)  # (B,168,128)


class AttentionLayer(nn.Module):
    """
    Soft attention over GRU hidden states.
    hidden_size must equal the GRU output dim:
      unidirectional → hidden_size
      bidirectional  → hidden_size * 2  ← used here
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, gru_out):
        weights = torch.softmax(self.attn(gru_out).squeeze(-1), dim=1)  # (B, 168)
        context = torch.bmm(weights.unsqueeze(1), gru_out).squeeze(1)   # (B, H)
        return context, weights


class SharedEncoder(nn.Module):
    """
    CNN → Bidirectional GRU → Attention.
    Output: context (B, hidden_size*2), attn_weights (B, 168).
    Shared across all four pipeline stages.
    input_size is passed at construction time and reflects the current
    width of X (base + however many cascade columns have been appended).
    """
    def __init__(self, input_size, cnn_channels=(64, 128, 128),
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = CNNEncoder(input_size, cnn_channels)
        self.gru = nn.GRU(
            input_size=cnn_channels[-1],
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=True,
        )
        self.drop  = nn.Dropout(dropout)
        self.attn  = AttentionLayer(hidden_size * 2)

    def forward(self, x):
        out, _ = self.gru(self.cnn(x))
        return self.attn(self.drop(out))


class DayAheadDecoder(nn.Module):
    """
    Autoregressive decoder — unrolls 24 steps, feeding each
    prediction back as the next input.  Matches inference behaviour.
    """
    def __init__(self, hidden_size, output_steps=24):
        super().__init__()
        self.steps = output_steps
        self.gru   = nn.GRU(1, hidden_size, num_layers=1, batch_first=True)
        self.fc    = nn.Linear(hidden_size, 1)

    def forward(self, context):
        h   = context.unsqueeze(0)
        dec = torch.zeros(context.size(0), 1, 1, device=context.device)
        out = []
        for _ in range(self.steps):
            h_out, h = self.gru(dec, h)
            p = self.fc(h_out)     # (B,1,1)
            out.append(p.squeeze(-1))
            dec = p
        return torch.cat(out, dim=1)   # (B, 24)


# ── Stage 1a, 1b, 2, 3 — Day-Ahead Model ─────────────────────────────────────
class DayAheadModel(nn.Module):
    """
    Used for: gen_solar, gen_wind, total load actual, price day ahead.
    input_size is read from fold_data['X_train'].shape[1] at each stage.
    """
    def __init__(self, input_size, cnn_channels=(64, 128, 128),
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = SharedEncoder(input_size, cnn_channels, hidden_size, num_layers, dropout)
        self.decoder = DayAheadDecoder(hidden_size * 2, output_steps=24)

    def forward(self, x_hist):
        context, attn = self.encoder(x_hist)
        return self.decoder(context), attn


# ── Stage 4 — Real-Time Price Model ──────────────────────────────────────────
class PriceRealTimeModel(nn.Module):
    """
    Predicts price actual for the next single hour.

    Stream 1 — SharedEncoder over x_hist (168, input_size_extended):
        input_size_extended = base features
                            + gen_solar cascade prediction    (1 col)
                            + gen_wind cascade prediction     (1 col)
                            + total load cascade prediction   (1 col)
        The encoder produces a context vector of shape (B, hidden_size*2).

    Stream 2 — Dense head receives context concatenated with x_forecast:
        x_forecast = [forecast_solar_day_ahead,        (TSO)
                       forecast_wind_onshore_day_ahead, (TSO)
                       total_load_forecast,             (TSO)
                       price_day_ahead]                 (settled market)
        These complement the cascade predictions with information
        the upstream models do not have (meteorological models, grid data).
        The model learns to weight Stream 1 and Stream 2 jointly.
    """
    def __init__(self, input_size, n_forecast_features,
                 cnn_channels=(64, 128, 128),
                 hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = SharedEncoder(input_size, cnn_channels, hidden_size, num_layers, dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2 + n_forecast_features, 256),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x_hist, x_forecast):
        context, attn = self.encoder(x_hist)
        return self.head(torch.cat([context, x_forecast], dim=-1)), attn

In [ ]:
# ── Sanity check on base input size before any cascade ────────────────────────
_cfg = {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.3}
fd   = folds[0]

_da = DayAheadModel(input_size=INPUT_SIZE_BASE, **_cfg)
_rt = PriceRealTimeModel(input_size=INPUT_SIZE_BASE,
                          n_forecast_features=INPUT_SIZE_FC, **_cfg)

def _count(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"DayAheadModel params    : {_count(_da):,}")
print(f"PriceRealTimeModel params: {_count(_rt):,}")

with torch.no_grad():
    _xh  = torch.zeros(2, 168, INPUT_SIZE_BASE)
    _xfc = torch.zeros(2, INPUT_SIZE_FC)
    _p_da, _a_da = _da(_xh)
    _p_rt, _a_rt = _rt(_xh, _xfc)

print(f"\nDayAhead  output: {tuple(_p_da.shape)}   (expect (2, 24))")
print(f"Attention output: {tuple(_a_da.shape)}  (expect (2, 168))")
print(f"RealTime  output: {tuple(_p_rt.shape)}   (expect (2, 1))")
print(f"\nNote: input_size will grow by 1 per cascade stage.")
print(f"  After Stage 1a: input_size = {INPUT_SIZE_BASE + 1}")
print(f"  After Stage 1b: input_size = {INPUT_SIZE_BASE + 2}")
print(f"  After Stage 2 : input_size = {INPUT_SIZE_BASE + 3}  (used by Stages 3 & 4)")
del _da, _rt

## 10. Loss, Training & Evaluation Functions

In [ ]:
class CustomLoss(nn.Module):
    """MAE + Jensen-Shannon divergence + smoothness penalty."""
    def __init__(self, alpha=0.1, beta=0.01):
        super().__init__()
        self.alpha, self.beta = alpha, beta

    def forward(self, yp, yt):
        yp, yt = yp.squeeze(), yt.squeeze()
        mae    = torch.mean(torch.abs(yt - yp))
        ts, ps = torch.softmax(yt, 0), torch.softmax(yp, 0)
        m      = 0.5 * (ts + ps)
        jsd    = 0.5 * (torch.sum(ts * torch.log(ts/(m+1e-8)+1e-8))
                       + torch.sum(ps * torch.log(ps/(m+1e-8)+1e-8)))
        smooth = torch.mean((yp[1:] - yp[:-1])**2)
        return mae + self.alpha * jsd + self.beta * smooth

In [ ]:
def train_model(model, train_loader, val_loader,
                model_type='dayahead',
                num_epochs=50, lr=1e-3,
                loss_fn='mae', alpha=0.1, beta=0.01):
    """
    Generic training loop for all four pipeline stages.

    model_type 'dayahead' : batch = (X_hist, y_24hr)
               'realtime' : batch = (X_hist, X_forecast, y_scalar)
                            X_hist contains cascade columns (Stream 1)
                            X_forecast contains TSO forecasts (Stream 2)
    """
    model = model.to(device)
    crit  = (nn.L1Loss() if loss_fn == 'mae' else
             nn.MSELoss() if loss_fn == 'mse' else
             CustomLoss(alpha, beta))
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='min', patience=5, factor=0.5)

    best_val, best_wts = float('inf'), copy.deepcopy(model.state_dict())
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(num_epochs):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        tr_losses = []
        for batch in tqdm(train_loader,
                          desc=f"Ep {epoch+1:>3}/{num_epochs} [Train]",
                          leave=False):
            if model_type == 'dayahead':
                xb, yb   = [t.to(device) for t in batch]
                pred, _  = model(xb)
                loss     = crit(pred, yb)
            else:
                xb, xfc, yb = [t.to(device) for t in batch]
                yb      = yb.unsqueeze(-1)
                pred, _ = model(xb, xfc)
                loss    = crit(pred, yb)

            if torch.isnan(loss) or torch.isinf(loss):
                continue
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_losses.append(loss.item())

        avg_tr = float(np.mean(tr_losses)) if tr_losses else float('nan')

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        va_losses = []
        with torch.no_grad():
            for batch in tqdm(val_loader,
                              desc=f"Ep {epoch+1:>3}/{num_epochs} [Val]  ",
                              leave=False):
                if model_type == 'dayahead':
                    xb, yb   = [t.to(device) for t in batch]
                    pred, _  = model(xb)
                    loss     = crit(pred, yb)
                else:
                    xb, xfc, yb = [t.to(device) for t in batch]
                    yb      = yb.unsqueeze(-1)
                    pred, _ = model(xb, xfc)
                    loss    = crit(pred, yb)

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    va_losses.append(loss.item())

        avg_va = float(np.mean(va_losses)) if va_losses else float('nan')

        if not np.isnan(avg_va) and avg_va < best_val:
            best_val = avg_va
            best_wts = copy.deepcopy(model.state_dict())
            print(f"  ✅ Ep {epoch+1:>3} — new best  val={best_val:.4f}")

        if not np.isnan(avg_va):
            sched.step(avg_va)

        history['train_loss'].append(avg_tr)
        history['val_loss'].append(avg_va)
        history['lr'].append(opt.param_groups[0]['lr'])
        print(f"Ep {epoch+1:>3}/{num_epochs}  train={avg_tr:.4f}  val={avg_va:.4f}  "
              f"lr={opt.param_groups[0]['lr']:.2e}")

    model.load_state_dict(best_wts)
    print(f"Training complete — best val loss: {best_val:.4f}")
    return model, history

In [ ]:
def evaluate_model(model, loader, target_scaler,
                   model_type='dayahead', split_name='val'):
    """
    Evaluate on any loader split.  Returns inverse-transformed arrays
    and a metrics dict.

    Call with val loader during HP tuning.
    Call with test loader once only after HPs are locked.
    """
    model.eval()
    all_preds, all_acts = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluating [{split_name}]", leave=True):
            if model_type == 'dayahead':
                xb, yb   = [t.to(device) for t in batch]
                pred, _  = model(xb)
            else:
                xb, xfc, yb = [t.to(device) for t in batch]
                yb      = yb.unsqueeze(-1)
                pred, _ = model(xb, xfc)

            all_preds.append(pred.cpu().numpy())
            all_acts.append(yb.cpu().numpy())

    preds = np.concatenate(all_preds, axis=0)
    acts  = np.concatenate(all_acts,  axis=0)

    if model_type == 'dayahead':
        n = preds.shape[0]
        preds_orig = target_scaler.inverse_transform(preds.reshape(-1,1)).reshape(n,24)
        acts_orig  = target_scaler.inverse_transform(acts.reshape(-1,1)).reshape(n,24)
    else:
        preds_orig = target_scaler.inverse_transform(preds.reshape(-1,1))
        acts_orig  = target_scaler.inverse_transform(acts.reshape(-1,1))

    mae     = float(np.mean(np.abs(acts_orig - preds_orig)))
    rmse    = float(np.sqrt(np.mean((acts_orig - preds_orig)**2)))
    rng     = float(acts_orig.max() - acts_orig.min())
    mae_pct = (mae / rng * 100) if rng > 0 else float('nan')

    print(f"  [{split_name}]  MAE={mae:.4f}  RMSE={rmse:.4f}  MAE%={mae_pct:.1f}%")
    return preds_orig, acts_orig, {'mae': mae, 'rmse': rmse,
                                    'mae_pct': mae_pct, 'range': rng}

## 11. Cascade Prediction Helper

`append_cascade_to_fold()` runs a trained Stage 1/2 model over every split,
extracts its single-step-ahead prediction at each row, scales it to [0,1],
and appends it as a new column to `fold_data['X_train/val/test']`.

After this call the next stage's `make_loaders_dayahead()` or
`make_loaders_realtime()` will automatically produce batches with the
wider X matrix — both `DayAheadDataset` and `RealTimeDataset` adapt to
the new width without any modification.

In [ ]:
def _generate_single_step_preds(model, X_split, y_split,
                                  window_size=WINDOW_SIZE, horizon=HORIZON_DA,
                                  batch_size=BATCH_SIZE):
    """
    Slide the model over X_split and collect one predicted value per row.

    For each window starting at index i the model produces a 24-hr forecast.
    We keep only the FIRST predicted hour (t+1) as the single-step estimate
    for row i+window_size.  The leading window_size rows are back-filled
    with the first available prediction.

    Returns: np.ndarray of shape (len(X_split),) — one value per row,
             in the same scaled space as the model outputs ([0, 1]).
    """
    dataset = DayAheadDataset(X_split, y_split, window_size, horizon)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=0)
    model.eval()
    step_preds = []

    with torch.no_grad():
        for xb, _ in loader:
            pred, _ = model(xb.to(device))   # (batch, 24)
            step_preds.append(pred[:, 0].cpu().numpy())  # keep hour-1 only

    step_preds = np.concatenate(step_preds)  # (N - window - horizon + 1,)

    # Pad front so output length == len(X_split)
    pad     = np.full(window_size + horizon - 1, step_preds[0])
    aligned = np.concatenate([pad, step_preds])[:len(X_split)]
    return aligned.astype(np.float32)


def append_cascade_to_fold(fold_data, target, trained_model,
                             window_size=WINDOW_SIZE):
    """
    Append upstream model predictions as a new feature column to
    fold_data['X_train'], fold_data['X_val'], and fold_data['X_test'].

    The appended values are the model's own [0,1]-scaled predictions,
    produced independently for each split using the split's own X array.
    This means:
      - Train split: model sees training-distribution inputs
      - Val split  : model sees val-distribution inputs (no leakage)
      - Test split : model sees test-distribution inputs (no leakage)

    After this call the downstream stage automatically receives the
    extended X when make_loaders_dayahead / make_loaders_realtime is called.
    """
    for split in ('train', 'val', 'test'):
        X_split = fold_data[f'X_{split}']
        y_split = fold_data[f'y_{split}'][target]

        preds = _generate_single_step_preds(
            trained_model, X_split, y_split, window_size=window_size)

        fold_data[f'X_{split}'] = np.hstack(
            [X_split, preds.reshape(-1, 1)])

    old_w = fold_data['X_train'].shape[1] - 1
    new_w = fold_data['X_train'].shape[1]
    fold_data.setdefault('cascade_preds', {})[target] = True
    print(f"  Cascade appended '{target}': X width {old_w} → {new_w}")


def reset_cascade(fold_data):
    """Reset working X arrays to base before each HP config."""
    fold_data['X_train']      = fold_data['X_train_base'].copy()
    fold_data['X_val']        = fold_data['X_val_base'].copy()
    fold_data['X_test']       = fold_data['X_test_base'].copy()
    fold_data['cascade_preds'] = {}
    print(f"Cascade reset — X width back to {fold_data['X_train'].shape[1]}")

## 12. Visualisation Helper

In [ ]:
def plot_results(preds, actuals, history, stage_label, target_name, split_name):
    is_da  = (preds.ndim == 2 and preds.shape[1] == 24)
    nrows  = 3 if is_da else 2
    fig    = plt.figure(figsize=(16, 5 * (nrows + 1)))
    fig.suptitle(f'{stage_label} — {target_name}  [{split_name}]',
                 fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(nrows + 1, 2, figure=fig, hspace=0.5, wspace=0.3)

    fp, fa = preds.flatten(), actuals.flatten()

    ax = fig.add_subplot(gs[0, :])
    ax.plot(fa, label='Actual',    color='steelblue', alpha=0.7, lw=0.8)
    ax.plot(fp, label='Predicted', color='orange',    alpha=0.7, lw=0.8)
    ax.set_title(f'Full {split_name} Period'); ax.set_xlabel('Hour index')
    ax.set_ylabel('Value'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = fig.add_subplot(gs[1, 0])
    ax.plot(fa[:168], color='steelblue', label='Actual',    alpha=0.7)
    ax.plot(fp[:168], color='orange',    label='Predicted', alpha=0.7)
    ax.set_title('First 7 Days'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = fig.add_subplot(gs[1, 1])
    ax.scatter(fa, fp, alpha=0.2, color='steelblue', s=4)
    lo, hi = min(fa.min(), fp.min()), max(fa.max(), fp.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect')
    ax.set_title('Scatter'); ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.legend(); ax.grid(True, alpha=0.3)

    if is_da:
        for j, (si, st) in enumerate([(0,'Sample Day 1'),(100,'Sample Day 100')]):
            si = min(si, len(preds)-1)
            ax = fig.add_subplot(gs[2, j])
            ax.plot(range(24), actuals[si], color='steelblue', marker='o',
                    markersize=4, label='Actual')
            ax.plot(range(24), preds[si],   color='orange',    marker='x',
                    markersize=4, label='Predicted')
            ax.set_title(f'24hr Profile — {st}')
            ax.set_xlabel('Hour'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = fig.add_subplot(gs[nrows, :])
    ep = range(1, len(history['train_loss'])+1)
    ax.plot(ep, history['train_loss'], color='steelblue', lw=1.5, label='Train')
    ax.plot(ep, history['val_loss'],   color='orange',    lw=1.5, label='Val')
    ax.set_title('Loss Curves'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

## 13. Hyperparameter Configurations

**Evaluation discipline:**
- All HP search uses the **validation set only**
- Test set is evaluated **once** after best config is selected
- Best config is chosen by price day-ahead validation MAE (adjust as needed)

In [ ]:
HP_CONFIGS = [
    {'hidden_size': 128, 'dropout': 0.2, 'lr': 1e-3, 'loss_fn': 'mae'},
    {'hidden_size': 256, 'dropout': 0.3, 'lr': 1e-3, 'loss_fn': 'mae'},
    {'hidden_size': 256, 'dropout': 0.3, 'lr': 5e-4, 'loss_fn': 'mae'},
    {'hidden_size': 512, 'dropout': 0.4, 'lr': 1e-3, 'loss_fn': 'mae'},
]

NUM_EPOCHS = 50
FOLD_IDX   = 0
fold_data  = folds[FOLD_IDX]
fn         = fold_data['fold']

STAGE_ORDER = [
    ('gen_solar',         'Stage 1a — gen_solar',        'dayahead'),
    ('gen_wind',          'Stage 1b — gen_wind',          'dayahead'),
    ('total load actual', 'Stage 2  — total load actual', 'dayahead'),
    (PRICE_DA_TARGET,     'Stage 3  — price day ahead',   'dayahead'),
    (PRICE_RT_TARGET,     'Stage 4  — price actual (RT)', 'realtime'),
]

print(f"Fold {fn}: train={fold_data['train_years']}  "
      f"val={fold_data['val_years']}  test={fold_data['test_years']} (held out)")
print(f"HP configs to evaluate: {len(HP_CONFIGS)}")
print(f"Stages per config     : {len(STAGE_ORDER)}")

## 14. Hyperparameter Search — Validation Set Only

For each config the full four-stage cascade is run end-to-end:
1. Reset X arrays to base (remove any cascade columns from previous config)
2. Train Stage 1a, append gen_solar predictions to X
3. Train Stage 1b, append gen_wind predictions to X
4. Train Stage 2, append load predictions to X
5. Train Stage 3 (price day ahead) on the now fully-extended X
6. Train Stage 4 (price real-time) — Stream 1 uses extended X, Stream 2 uses fixed Xfc

Validation metrics are recorded for each config. Test set not touched.

In [ ]:
hp_results = []

for cfg_idx, cfg in enumerate(HP_CONFIGS):
    print(f"\n{'='*72}")
    print(f"Config {cfg_idx+1}/{len(HP_CONFIGS)}: {cfg}")
    print(f"{'='*72}")

    # ── Reset cascade so previous config's columns don't carry over ───────────
    reset_cascade(fold_data)

    record = {'config': cfg, 'val_metrics': {}, 'models': {}, 'histories': {}}

    # ── Stage 1a: gen_solar ───────────────────────────────────────────────────
    print(f"\n── Stage 1a: gen_solar  (X width={fold_data['X_train'].shape[1]}) ──")
    tr, va, _ = make_loaders_dayahead(fold_data, 'gen_solar')
    m = DayAheadModel(input_size=fold_data['X_train'].shape[1],
                       hidden_size=cfg['hidden_size'], num_layers=NUM_LAYERS,
                       dropout=cfg['dropout'])
    m, h = train_model(m, tr, va, model_type='dayahead',
                        num_epochs=NUM_EPOCHS, lr=cfg['lr'], loss_fn=cfg['loss_fn'])
    p, a, metrics = evaluate_model(m, va, fold_data['target_scalers']['gen_solar'],
                                    model_type='dayahead', split_name='val')
    record['val_metrics']['gen_solar']  = metrics
    record['models']['gen_solar']       = m
    record['histories']['gen_solar']    = h
    # Append gen_solar predictions — X width increases by 1
    append_cascade_to_fold(fold_data, 'gen_solar', m)

    # ── Stage 1b: gen_wind ────────────────────────────────────────────────────
    print(f"\n── Stage 1b: gen_wind  (X width={fold_data['X_train'].shape[1]}) ──")
    tr, va, _ = make_loaders_dayahead(fold_data, 'gen_wind')
    m = DayAheadModel(input_size=fold_data['X_train'].shape[1],
                       hidden_size=cfg['hidden_size'], num_layers=NUM_LAYERS,
                       dropout=cfg['dropout'])
    m, h = train_model(m, tr, va, model_type='dayahead',
                        num_epochs=NUM_EPOCHS, lr=cfg['lr'], loss_fn=cfg['loss_fn'])
    p, a, metrics = evaluate_model(m, va, fold_data['target_scalers']['gen_wind'],
                                    model_type='dayahead', split_name='val')
    record['val_metrics']['gen_wind']  = metrics
    record['models']['gen_wind']       = m
    record['histories']['gen_wind']    = h
    # Append gen_wind predictions — X width increases by 1
    append_cascade_to_fold(fold_data, 'gen_wind', m)

    # ── Stage 2: total load actual ────────────────────────────────────────────
    print(f"\n── Stage 2: total load  (X width={fold_data['X_train'].shape[1]}) ──")
    tr, va, _ = make_loaders_dayahead(fold_data, 'total load actual')
    m = DayAheadModel(input_size=fold_data['X_train'].shape[1],
                       hidden_size=cfg['hidden_size'], num_layers=NUM_LAYERS,
                       dropout=cfg['dropout'])
    m, h = train_model(m, tr, va, model_type='dayahead',
                        num_epochs=NUM_EPOCHS, lr=cfg['lr'], loss_fn=cfg['loss_fn'])
    p, a, metrics = evaluate_model(m, va, fold_data['target_scalers']['total load actual'],
                                    model_type='dayahead', split_name='val')
    record['val_metrics']['total load actual']  = metrics
    record['models']['total load actual']       = m
    record['histories']['total load actual']    = h
    # Append load predictions — X width increases by 1
    append_cascade_to_fold(fold_data, 'total load actual', m)

    # ── Stage 3: price day ahead ──────────────────────────────────────────────
    # X now includes gen_solar + gen_wind + load cascade columns
    print(f"\n── Stage 3: price day ahead  (X width={fold_data['X_train'].shape[1]}) ──")
    tr, va, _ = make_loaders_dayahead(fold_data, PRICE_DA_TARGET)
    m = DayAheadModel(input_size=fold_data['X_train'].shape[1],
                       hidden_size=cfg['hidden_size'], num_layers=NUM_LAYERS,
                       dropout=cfg['dropout'])
    m, h = train_model(m, tr, va, model_type='dayahead',
                        num_epochs=NUM_EPOCHS, lr=cfg['lr'], loss_fn=cfg['loss_fn'])
    p, a, metrics = evaluate_model(m, va, fold_data['target_scalers'][PRICE_DA_TARGET],
                                    model_type='dayahead', split_name='val')
    record['val_metrics'][PRICE_DA_TARGET]  = metrics
    record['models'][PRICE_DA_TARGET]       = m
    record['histories'][PRICE_DA_TARGET]    = h

    # ── Stage 4: price actual (real-time) ─────────────────────────────────────
    # Stream 1 (X_hist)    : extended X with gen + load cascade columns
    # Stream 2 (X_forecast): fixed Xfc with TSO forecasts + day-ahead price
    print(f"\n── Stage 4: price actual RT  (X width={fold_data['X_train'].shape[1]}, "
          f"Stream2 width={INPUT_SIZE_FC}) ──")
    tr_rt, va_rt, _ = make_loaders_realtime(fold_data)
    m = PriceRealTimeModel(input_size=fold_data['X_train'].shape[1],
                            n_forecast_features=INPUT_SIZE_FC,
                            hidden_size=cfg['hidden_size'], num_layers=NUM_LAYERS,
                            dropout=cfg['dropout'])
    m, h = train_model(m, tr_rt, va_rt, model_type='realtime',
                        num_epochs=NUM_EPOCHS, lr=cfg['lr'], loss_fn=cfg['loss_fn'])
    p, a, metrics = evaluate_model(m, va_rt, fold_data['target_scalers'][PRICE_RT_TARGET],
                                    model_type='realtime', split_name='val')
    record['val_metrics'][PRICE_RT_TARGET]  = metrics
    record['models'][PRICE_RT_TARGET]       = m
    record['histories'][PRICE_RT_TARGET]    = h

    hp_results.append(record)
    print(f"\n✅ Config {cfg_idx+1} complete")

print(f"\n✅ HP search complete — {len(hp_results)} configs evaluated")

## 15. Validation Results — Hyperparameter Comparison

In [ ]:
print(f"\n{'='*100}")
print("VALIDATION RESULTS — ALL CONFIGS")
print(f"{'='*100}")

for target, stage_label, _ in STAGE_ORDER:
    print(f"\n{stage_label}:")
    print(f"  {'Config':<55} {'MAE':>8} {'RMSE':>8} {'MAE%':>7}  Status")
    print(f"  {'-'*85}")
    for r in hp_results:
        m      = r['val_metrics'][target]
        status = '✅' if m['mae_pct'] < 10 else ('⚠️' if m['mae_pct'] < 20 else '❌')
        print(f"  {str(r['config']):<55} "
              f"{m['mae']:>8.3f}  {m['rmse']:>8.3f}  {m['mae_pct']:>6.1f}%  {status}")

In [ ]:
# ── Select best config ────────────────────────────────────────────────────────
# Primary sort: price day-ahead val MAE.
# Tiebreak: price real-time val MAE.
best_result = min(
    hp_results,
    key=lambda r: (r['val_metrics'][PRICE_DA_TARGET]['mae'],
                   r['val_metrics'][PRICE_RT_TARGET]['mae'])
)
best_cfg = best_result['config']

print(f"Best config: {best_cfg}")
print(f"\nVal metrics for best config:")
for target, stage_label, _ in STAGE_ORDER:
    m = best_result['val_metrics'][target]
    s = '✅' if m['mae_pct'] < 10 else ('⚠️' if m['mae_pct'] < 20 else '❌')
    print(f"  {stage_label:<38} MAE={m['mae']:.3f}  RMSE={m['rmse']:.3f}  "
          f"MAE%={m['mae_pct']:.1f}%  {s}")

## 16. Validation Visualisations — Best Config

In [ ]:
# Rebuild cascade for best config so X arrays are correctly extended
# before we call evaluate_model for each stage.
reset_cascade(fold_data)

best_models    = best_result['models']
best_histories = best_result['histories']

for target, stage_label, mtype in STAGE_ORDER:
    model   = best_models[target]
    history = best_histories[target]

    if mtype == 'realtime':
        _, va_rt, _ = make_loaders_realtime(fold_data)
        preds, acts, _ = evaluate_model(
            model, va_rt, fold_data['target_scalers'][target],
            model_type='realtime', split_name='val')
    else:
        _, va, _ = make_loaders_dayahead(fold_data, target)
        preds, acts, _ = evaluate_model(
            model, va, fold_data['target_scalers'][target],
            model_type='dayahead', split_name='val')
        # Append cascade for non-terminal stages
        if target in CASCADE_TARGETS:
            append_cascade_to_fold(fold_data, target, model)

    plot_results(preds, acts, history,
                 stage_label=stage_label,
                 target_name=target,
                 split_name='val')

## 17. Final Test Evaluation

**Run once only.** HPs are now locked. Do not adjust anything after seeing these numbers.

In [ ]:
# Cascade should already be appended from the visualisation cell above.
# Verify widths are consistent before touching the test set.
expected_width = INPUT_SIZE_BASE + len(CASCADE_TARGETS)
for split in ('train', 'val', 'test'):
    actual = fold_data[f'X_{split}'].shape[1]
    assert actual == expected_width, (
        f"X_{split} width {actual} != expected {expected_width}. "
        f"Re-run Section 16 (validation visualisations) to rebuild cascade.")

print(f"Feature width check passed: {expected_width} columns in all splits")
print(f"  Base features   : {INPUT_SIZE_BASE}")
print(f"  Cascade columns : {len(CASCADE_TARGETS)} ({CASCADE_TARGETS})")
print(f"  Stream 2 width  : {INPUT_SIZE_FC} (unchanged, TSO forecasts)")

test_results = {}

for target, stage_label, mtype in STAGE_ORDER:
    print(f"\n── {stage_label} ──")
    model = best_models[target]

    if mtype == 'realtime':
        _, _, te_rt = make_loaders_realtime(fold_data)
        preds, acts, metrics = evaluate_model(
            model, te_rt, fold_data['target_scalers'][target],
            model_type='realtime', split_name='test')
    else:
        _, _, te = make_loaders_dayahead(fold_data, target)
        preds, acts, metrics = evaluate_model(
            model, te, fold_data['target_scalers'][target],
            model_type='dayahead', split_name='test')

    test_results[target] = {'preds': preds, 'actuals': acts, 'metrics': metrics}

# ── Summary table ──────────────────────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"FINAL TEST RESULTS  —  Best Config: {best_cfg}")
print(f"{'='*75}")
print(f"\n{'Stage':<42} {'MAE':>8} {'RMSE':>8} {'MAE%':>7}  Status")
print("-" * 75)

for target, stage_label, _ in STAGE_ORDER:
    m = test_results[target]['metrics']
    s = '✅' if m['mae_pct'] < 10 else ('⚠️' if m['mae_pct'] < 20 else '❌')
    print(f"{stage_label:<42} {m['mae']:>8.3f}  {m['rmse']:>8.3f}  "
          f"{m['mae_pct']:>6.1f}%  {s}")

## 18. Test Set Visualisations

In [ ]:
for target, stage_label, _ in STAGE_ORDER:
    r = test_results[target]
    plot_results(r['preds'], r['actuals'], best_histories[target],
                 stage_label=stage_label,
                 target_name=target,
                 split_name='test')

## 19. Save Best Models

In [ ]:
os.makedirs('models', exist_ok=True)

# input_size per stage reflects X width at time that stage was trained
stage_input_sizes = {
    'gen_solar'         : INPUT_SIZE_BASE,
    'gen_wind'          : INPUT_SIZE_BASE + 1,
    'total load actual' : INPUT_SIZE_BASE + 2,
    PRICE_DA_TARGET     : INPUT_SIZE_BASE + 3,
    PRICE_RT_TARGET     : INPUT_SIZE_BASE + 3,
}

save_paths = {
    'gen_solar'         : 'models/stage1a_gen_solar.pt',
    'gen_wind'          : 'models/stage1b_gen_wind.pt',
    'total load actual' : 'models/stage2_load.pt',
    PRICE_DA_TARGET     : 'models/stage3_price_dayahead.pt',
    PRICE_RT_TARGET     : 'models/stage4_price_realtime.pt',
}

for target, stage_label, mtype in STAGE_ORDER:
    ckpt = {
        'model_state'       : best_models[target].state_dict(),
        'target'            : target,
        'model_type'        : mtype,
        'best_config'       : best_cfg,
        'input_size'        : stage_input_sizes[target],
        'scaler_X'          : fold_data['scaler_X'],
        'target_scaler'     : fold_data['target_scalers'][target],
        'base_feature_cols' : BASE_HIST_FEATURE_COLS,
        'cascade_targets'   : CASCADE_TARGETS,
        'window_size'       : WINDOW_SIZE,
        'val_metrics'       : best_result['val_metrics'][target],
        'test_metrics'      : test_results[target]['metrics'],
    }
    if mtype == 'realtime':
        ckpt['scaler_forecast']     = fold_data['scaler_forecast']
        ckpt['forecast_cols']       = REALTIME_FORECAST_COLS
        ckpt['n_forecast_features'] = INPUT_SIZE_FC

    torch.save(ckpt, save_paths[target])
    print(f"Saved {stage_label} → {save_paths[target]}")